# Portfolio Tear Sheet

Build a small multi-instrument, multi-entity portfolio, value it, aggregate metrics
and cashflows, then render a `portfolio_tearsheet`.

In [1]:
import datetime as dt
import json
from datetime import date

from finstack_quant import reporting
from finstack_quant.core.market_data import DiscountCurve, MarketContext
from finstack_quant.portfolio import aggregate_full_cashflows, aggregate_metrics, value_portfolio

# ----- Market context -----
mc = MarketContext()
mc.insert(DiscountCurve(
    "USD-OIS",
    date(2025, 1, 15),
    [(0.0, 1.0), (0.25, 0.9875), (0.5, 0.975), (1.0, 0.95), (2.0, 0.90), (5.0, 0.75)],
    day_count="act_365f",
))
market_json = mc.to_json()
print("Market context built; curves:", [c["id"] for c in json.loads(market_json)["curves"]])


Market context built; curves: ['USD-OIS']


## Portfolio spec — four deposits across two entities

In [2]:
def dep_pos(pos_id, entity_id, instr_id, notional, rate, maturity):
    return {
        "position_id": pos_id,
        "entity_id": entity_id,
        "instrument_id": instr_id,
        "instrument_spec": {
            "type": "deposit",
            "spec": {
                "id": instr_id,
                "notional": {"amount": notional, "currency": "USD"},
                "start_date": "2025-01-15",
                "maturity": maturity,
                "day_count": "Act360",
                "quote_rate": rate,
                "discount_curve_id": "USD-OIS",
                "attributes": {},
            },
        },
        "quantity": 1.0,
        "unit": "units",
    }

portfolio_spec = {
    "id": "multi-asset-book",
    "as_of": "2025-01-15",
    "base_ccy": "USD",
    "entities": {
        "FUND-1": {"id": "FUND-1"},
        "FUND-2": {"id": "FUND-2"},
    },
    "positions": [
        dep_pos("F1-3M",  "FUND-1", "DEP-3M",  5_000_000.0, 0.045, "2025-04-15"),
        dep_pos("F1-6M",  "FUND-1", "DEP-6M",  8_000_000.0, 0.050, "2025-07-15"),
        dep_pos("F1-1Y",  "FUND-1", "DEP-1Y",  3_000_000.0, 0.055, "2026-01-15"),
        dep_pos("F2-3M",  "FUND-2", "DEP-3M-B", 4_000_000.0, 0.042, "2025-04-15"),
        dep_pos("F2-6M",  "FUND-2", "DEP-6M-B", 6_000_000.0, 0.048, "2025-07-15"),
        dep_pos("F2-2Y",  "FUND-2", "DEP-2Y",  2_000_000.0, 0.058, "2027-01-15"),
    ],
}
portfolio_json = json.dumps(portfolio_spec)
print(f"Portfolio: {len(portfolio_spec["positions"])} positions across {len(portfolio_spec["entities"])} entities")


Portfolio: 6 positions across 2 entities


## Valuation, metrics, and cashflows

In [3]:
valuation = value_portfolio(portfolio_json, market_json)
metrics = aggregate_metrics(valuation, "USD", market_json, "2025-01-15")
cf_obj = aggregate_full_cashflows(portfolio_json, market_json)
cashflows = cf_obj.to_json()

val_parsed = json.loads(valuation)
total_pv = float(val_parsed["total_base_ccy"]["amount"])
print(f"Total PV: {total_pv:,.2f} USD")
print(f"Entities: {list(val_parsed["by_entity"].keys())}")


Total PV: 27,996,918.40 USD
Entities: ['FUND-1', 'FUND-2']


## Book summary

In [4]:
reporting.portfolio_tearsheet(
    valuation,
    metrics=metrics,
    cashflows=cashflows,
    title="Multi-Asset Book",
    generated=dt.date(2026, 6, 23),
)

TearSheet(theme=Theme(name='institutional', font_head='Georgia,"Times New Roman",serif', font_num='"Iowan Old Style",Georgia,serif', font_sans="-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif", ink='#10243f', muted='#7a8190', pos='#1b6b4f', neg='#9b2335', accent='#7c2230', canvas='#fbfaf6', rule='#10243f', faint='#e4e1d8', grid='#cfc8b8'), title='Multi-Asset Book', sections=[Section(title='Holdings', body='<table class="dd"><thead><tr><th>Position</th><th>Entity</th><th>Native</th><th>Base</th></tr></thead><tbody><tr><td class="">F1-6M</td><td class="">FUND-1</td><td class="">7,997,768.67 USD</td><td class="">7,997,768.67</td></tr><tr><td class="">F2-6M</td><td class="">FUND-2</td><td class="">5,992,442.76 USD</td><td class="">5,992,442.76</td></tr><tr><td class="">F1-3M</td><td class="">FUND-1</td><td class="">4,993,912.72 USD</td><td class="">4,993,912.72</td></tr><tr><td class="">F2-3M</td><td class="">FUND-2</td><td class="">3,992,167.16 USD</td><td class="">3,992,167.16</td></tr><tr><td class="">F1-1Y</td><td class="">FUND-1</td><td class="">3,008,927.08 USD</td><td class="">3,008,927.08</td></tr><tr><td class="">F2-2Y</td><td class="">FUND-2</td><td class="">2,011,700.00 USD</td><td class="">2,011,700.00</td></tr></tbody></table>', subtitle=None), Section(title='Exposure by Entity', body='<div class="grid2"><div><svg viewBox="0 0 620 190" xmlns="http://www.w3.org/2000/svg"><line x1="48" y1="166.0" x2="606" y2="166.0" stroke="#cfc8b8"/><text x="42" y="169.0" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">0</text><line x1="48" y1="127.5" x2="606" y2="127.5" stroke="#e4e1d8"/><text x="42" y="130.5" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">5000000</text><line x1="48" y1="89.0" x2="606" y2="89.0" stroke="#e4e1d8"/><text x="42" y="92.0" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">10000000</text><line x1="48" y1="50.5" x2="606" y2="50.5" stroke="#e4e1d8"/><text x="42" y="53.5" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">15000000</text><line x1="48" y1="12.0" x2="606" y2="12.0" stroke="#e4e1d8"/><text x="42" y="15.0" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">20000000</text><rect class="fq-hb" x="101.0" y="42.8" width="173.0" height="123.2" fill="#1b6b4f" fill-opacity="0.82" data-cx="187.5" data-cy="42.8" data-label="FUND-1" data-val="16000608.47"><title>FUND-1 · 16000608.47</title></rect><text x="187.5" y="181" text-anchor="middle" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">FUND-1</text><text x="187.5" y="38.8" text-anchor="middle" font-size="9.5" fill="#23303f" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">16,000,608</text><rect class="fq-hb" x="380.0" y="73.6" width="173.0" height="92.4" fill="#1b6b4f" fill-opacity="0.82" data-cx="466.5" data-cy="73.6" data-label="FUND-2" data-val="11996309.92"><title>FUND-2 · 11996309.92</title></rect><text x="466.5" y="181" text-anchor="middle" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">FUND-2</text><text x="466.5" y="69.6" text-anchor="middle" font-size="9.5" fill="#23303f" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">11,996,310</text><line class="fq-cross" x1="0" x2="0" y1="12" y2="166" style="visibility:hidden" pointer-events="none"/><circle class="fq-mk" r="3.5" cx="0" cy="0" style="visibility:hidden" pointer-events="none"/></svg></div><div><table class="dd"><thead><tr><th>Entity</th><th>Base</th></tr></thead><tbody><tr><td class="">FUND-1</td><td class="">16,000,608.47</td></tr><tr><td class="">FUND-2</td><td class="">11,996,309.92</td></tr></tbody></table></div></div>', subtitle=None), Section(title='Aggregated Sensitivities', body='<table class="dd"><thead><tr><th>Metri

## Saving a standalone HTML file

```python
ts = reporting.portfolio_tearsheet(valuation, metrics=metrics, cashflows=cashflows, generated=dt.date(2026, 6, 23))
ts.save("portfolio_tearsheet.html")
```